In [1]:

from src.pipeline.settings import Settings
from src.pipeline.logging_config import get_logger

s = Settings()
log = get_logger()
log.info(f'''smoke test — settings: {s.model_dump(mode='json')}''')
print('OK — settings constructed, log line written')

OK — settings constructed, log line written


In [1]:
## ### Step 2b — Fill in `ask_llm` · 💻 Self-paced · 10 min
# **5. Run it — call the function directly to confirm it works.**
#note that this snippet was provided to run from cmd. I replaced asyncio.run() with await
import asyncio
from src.pipeline.pipeline import ask_llm, Question
ans = await ask_llm(Question(text='What is RAG in one sentence?'))
print('type:', type(ans).__name__)
print('text:', ans.text[:80])
print('cost:', ans.cost_usd)
print('retries:', ans.retries)

type: Answer
text: RAG combines retrieval over a document corpus with an LLM, so answers are ground
cost: 0.0001
retries: 0


In [11]:
## ### Step 2b — Fill in `ask_llm` · 💻 Self-paced · 10 min
# **5. Run it — call the function directly to confirm it works.**
#note that this snippet was provided to run from cmd. I replaced asyncio.run() with await
import asyncio
from src.pipeline.pipeline import ask_llm, Question
ans = await ask_llm(Question(text='What is RAG in one sentence?'), fail_rate=0.5)
print('type:', type(ans).__name__)
print('text:', ans.text[:80])
print('cost:', ans.cost_usd)
print('retries:', ans.retries)

type: Answer
text: RAG combines retrieval over a document corpus with an LLM, so answers are ground
cost: 0.0001
retries: 0


In [16]:
import asyncio
from src.pipeline.pipeline import ask_llm_with_retry, Question

# Clean — should succeed on attempt 0, no retries
ans = await ask_llm_with_retry(Question(text='What is RAG?'))
print('clean:    retries =', ans.retries)

# Always-fail — should retry twice (3 attempts total) and then raise
try:
    await ask_llm_with_retry(Question(text='What is RAG?'), fail_rate=1.0)
except Exception as e:
    print('lossy:    raised after 3 attempts:', type(e).__name__)

clean:    retries = 0
lossy:    raised after 3 attempts: FakeLLMError


In [5]:
import asyncio, time
from src.pipeline.pipeline import run_batch, Question

questions = [
    Question(text='What is RAG?'),
    Question(text='Name three uses of vector databases.'),
    Question(text='Why might an LLM hallucinate?'),
]
t0 = time.time()
answers = await run_batch(questions)
elapsed = time.time() - t0
print(f'wall-clock: {elapsed:.2f}s')
for a in answers:
    print(f'- {a.text[:60]}')

wall-clock: 1.44s
- RAG combines retrieval over a document corpus with an LLM, s
- Vector databases power semantic search, RAG context retrieva
- LLMs hallucinate when they produce confident text that isn't


In [4]:
from src.pipeline.pipeline import load_questions
qs = load_questions()
print(len(qs))
print(qs[0])
print(type(qs[0]).__name__)

20
text='What is retrieval-augmented generation in one sentence?'
Question


In [4]:
from src.pipeline.settings import RunSummary; 
RunSummary(started_at=1.5, elapsed_seconds=1, n_questions=0, n_succeeded=0, n_retries_total=0, total_cost_usd=0, fail_rate=0, use_fake=True)

RunSummary(started_at=1.5, elapsed_seconds=1.0, n_questions=0, n_succeeded=0, n_retries_total=0, total_cost_usd=0.0, fail_rate=0.0, use_fake=True)

In [5]:
from src.pipeline.settings import RunSummary
import json
s = RunSummary(started_at=1717.0, elapsed_seconds=4.42, n_questions=20, n_succeeded=20, n_retries_total=0, total_cost_usd=0.002, fail_rate=0.0, use_fake=True)
print(json.dumps(s.model_dump(), indent=2))

{
  "started_at": 1717.0,
  "elapsed_seconds": 4.42,
  "n_questions": 20,
  "n_succeeded": 20,
  "n_retries_total": 0,
  "total_cost_usd": 0.002,
  "fail_rate": 0.0,
  "use_fake": true
}


In [7]:
from pathlib import Path
import os
from dotenv import load_dotenv
env_path = Path("../../practice_scripts/.env")
load_dotenv(dotenv_path=env_path)
print(os.getenv("OPENAI_API_KEY"))

voc-159143191421403752274466a86df03533da5.24484286


In [1]:
import sys
from pathlib import Path

# Add the project root (parent folder) to sys.path
sys.path.append(str(Path.cwd().parent))
from src.pipeline.store import connect
con = connect('test.db')
print(con.execute('SELECT name FROM sqlite_master WHERE type=\"table\"').fetchall())

[('runs',), ('sqlite_sequence',), ('answers',)]


In [2]:
import sqlite3
con = sqlite3.connect('results.db')
print('runs:')
for r in con.execute('SELECT id, n_questions, n_retries_total, total_cost_usd, fail_rate, use_fake FROM runs'):
    print(' ', r)
print('answers per run:')
for r in con.execute('SELECT run_id, COUNT(*) FROM answers GROUP BY run_id'):
    print(' ', r)

runs:
  (1, 20, 0, 0.002, 0.0, 0)
answers per run:
  (1, 20)
